In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')



Решаем уравнение Лапласа для потенциала

$$
\nabla^2 \varphi(x,y) = 0,
$$

где на пластинах и стенках задан потенциал, а внутри можно добавить проводящий квадрат.
Электрическое поле выражается через потенциал

$$
\vec{E} = -\nabla \varphi = \left(-\frac{\partial \varphi}{\partial x},
                                   -\frac{\partial \varphi}{\partial y}\right).
$$


In [ ]:
def create_parallel_plate_geometry(nx=121, ny=121, V0=1.0,
                                   with_square=False,
                                   square_V=0.0,
                                   square_size=21):

    phi = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros_like(phi, dtype=bool)
    phi_fix = np.zeros_like(phi, dtype=float)
    square_mask = np.zeros_like(phi, dtype=bool)

    # Верхняя пластина: φ = +V0
    fixed[-1, :] = True
    phi_fix[-1, :] = +V0

    # Нижняя пластина: φ = -V0
    fixed[0, :] = True
    phi_fix[0, :] = -V0

    # Боковые стенки: φ = 0
    fixed[:, 0] = True
    phi_fix[:, 0] = 0.0

    fixed[:, -1] = True
    phi_fix[:, -1] = 0.0

    if with_square:
        ic = nx // 2
        jc = ny // 2

        half = square_size // 2
        i_min = ic - half
        i_max = i_min + square_size
        j_min = jc - half
        j_max = j_min + square_size

        i_min = max(i_min, 1)
        j_min = max(j_min, 1)
        i_max = min(i_max, nx-1)
        j_max = min(j_max, ny-1)

        square_mask[j_min:j_max, i_min:i_max] = True
        fixed[j_min:j_max, i_min:i_max] = True
        phi_fix[j_min:j_max, i_min:i_max] = square_V

    phi[fixed] = phi_fix[fixed]

    return phi, fixed, phi_fix, square_mask


def compute_field(phi, dx=1.0, dy=1.0):
    dphidy, dphidx = np.gradient(phi, dy, dx)
    Ex = -dphidx
    Ey = -dphidy
    return Ex, Ey


def normalize_field(Ex, Ey, eps=1e-12):
    mag = np.sqrt(Ex**2 + Ey**2)
    mag = np.maximum(mag, eps)
    return Ex/mag, Ey/mag


### Блок 3. Итерационные методы

Используем конечно-разностную аппроксимацию уравнения Лапласа

$$
\varphi_{i,j} \approx \frac{1}{4}
\left(
\varphi_{i+1,j} + \varphi_{i-1,j} +
\varphi_{i,j+1} + \varphi_{i,j-1}
\right).
$$

**Метод Якоби:**

$$
\varphi^{(k+1)}_{i,j} =
\frac{1}{4}\left(
\varphi^{(k)}_{i+1,j} +
\varphi^{(k)}_{i-1,j} +
\varphi^{(k)}_{i,j+1} +
\varphi^{(k)}_{i,j-1}
\right).
$$

**Метод Гаусса–Зейделя:**

$$
\varphi^{(k+1)}_{i,j} =
\frac{1}{4}\left(
\varphi^{(k+1)}_{i-1,j} +
\varphi^{(k+1)}_{i,j-1} +
\varphi^{(k)}_{i+1,j} +
\varphi^{(k)}_{i,j+1}
\right).
$$

Контролируем невязку

$$
r^{(k)} = \max_{i,j} \left| \Delta \varphi^{(k)}_{i,j} \right|.
$$


In [ ]:
def solve_laplace_jacobi(phi_init, fixed,
                         tol=1e-5, max_iters=50000,
                         verbose=True, store_history=False):
    """Решение уравнения Лапласа методом Якоби.

    phi_init  - начальное поле потенциала (включая фиксированные узлы)
    fixed     - булев массив тех же размеров: True -> узел с заданным φ
    tol       - требуемая точность (по максимуму изменения φ за шаг)
    max_iters - ограничение на число итераций
    verbose   - печатать ли информацию о сходимости
    store_history - сохранять ли историю невязки

    Возвращает:
        phi       - найденное поле потенциала
        iters     - число сделанных итераций
        residual  - последняя невязка (max|Δφ|)
        history   - список невязок (если store_history=True, иначе None)
    """
    phi = phi_init.copy()
    ny, nx = phi.shape

    dx = dy = 1.0
    inv_denom = 1.0 / (2.0/dx**2 + 2.0/dy**2)

    history = [] if store_history else None

    for it in range(1, max_iters + 1):
        new_phi = phi.copy()
        max_delta = 0.0

        # обновляем только внутренние узлы, не выходя на границы
        for j in range(1, ny-1):
            for i in range(1, nx-1):
                if fixed[j, i]:
                    continue  # у фиксированных узлов φ не меняем

                num = (phi[j, i+1] + phi[j, i-1]) / dx**2 \
                    + (phi[j+1, i] + phi[j-1, i]) / dy**2
                val = num * inv_denom

                d = abs(val - phi[j, i])
                if d > max_delta:
                    max_delta = d

                new_phi[j, i] = val

        phi = new_phi

        if store_history:
            history.append(max_delta)

        if verbose and it % 1000 == 0:
            print(f"Якоби: итерация {it}, maxΔφ = {max_delta:.3e}")

        if max_delta < tol:
            if verbose:
                print(f"Якоби сошёлся за {it} итераций, maxΔφ = {max_delta:.3e}")
            return phi, it, max_delta, history

    if verbose:
        print(f"Якоби достиг max_iters={max_iters}, последняя maxΔφ = {max_delta:.3e}")
    return phi, max_iters, max_delta, history


def solve_laplace_gauss_seidel(phi_init, fixed,
                               tol=1e-5, max_iters=50000,
                               verbose=True, store_history=False):

    phi = phi_init.copy()
    ny, nx = phi.shape

    dx = dy = 1.0
    inv_denom = 1.0 / (2.0/dx**2 + 2.0/dy**2)

    history = [] if store_history else None

    for it in range(1, max_iters + 1):
        max_delta = 0.0

        for j in range(1, ny-1):
            for i in range(1, nx-1):
                if fixed[j, i]:
                    continue

                num = (phi[j, i+1] + phi[j, i-1]) / dx**2 \
                    + (phi[j+1, i] + phi[j-1, i]) / dy**2
                val = num * inv_denom

                d = abs(val - phi[j, i])
                if d > max_delta:
                    max_delta = d

                phi[j, i] = val

        if store_history:
            history.append(max_delta)

        if verbose and it % 1000 == 0:
            print(f"Гаусс-Зейдель: итерация {it}, maxΔφ = {max_delta:.3e}")

        if max_delta < tol:
            if verbose:
                print(f"Гаусс-Зейдель сошёлся за {it} итераций, maxΔφ = {max_delta:.3e}")
            return phi, it, max_delta, history

    if verbose:
        print(f"Гаусс-Зейдель достиг max_iters={max_iters}, последняя maxΔφ = {max_delta:.3e}")
    return phi, max_iters, max_delta, history


### Визуализация

Показываем карту потенциала$$\ \varphi(x,y) \$$ и линии/векторы поля

$$
\vec{E}(x,y) = -\nabla \varphi(x,y).
$$

Также строим график сходимости $$\ r^{(k)} = \max | \Delta \varphi^{(k)}_{i,j} | \$$.


In [ ]:
def plot_potential(phi, title='Карта потенциала φ', square_mask=None):
    ny, nx = phi.shape
    x = np.arange(nx)
    y = np.arange(ny)

    plt.figure(figsize=(7, 6))
    im = plt.imshow(phi, origin='lower',
                    extent=[x.min(), x.max()-1, y.min(), y.max()-1],
                    aspect='equal', cmap='RdBu_r')
    plt.colorbar(im, label='Потенциал φ')

    if square_mask is not None and square_mask.any():

        sm = square_mask.astype(float)
        plt.contour(x, y, sm, levels=[0.5], colors='k', linewidths=2)

    plt.title(title)
    plt.xlabel('x (узел)')
    plt.ylabel('y (узел)')
    plt.tight_layout()
    plt.show()


def plot_field_quiver(Ex, Ey, title='Векторное поле E',
                      step=3):
    ny, nx = Ex.shape
    x = np.arange(nx)
    y = np.arange(ny)
    X, Y = np.meshgrid(x, y)

    U, V = normalize_field(Ex, Ey)

    plt.figure(figsize=(7, 6))
    skip = (slice(None, None, step), slice(None, None, step))
    plt.quiver(X[skip], Y[skip], U[skip], V[skip],
               pivot='mid', scale_units='xy', angles='xy', scale=1.0)
    plt.title(title)
    plt.xlabel('x (узел)')
    plt.ylabel('y (узел)')
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.show()


def plot_field_stream(Ex, Ey, title='Линии поля E'):
    ny, nx = Ex.shape
    x = np.arange(nx)
    y = np.arange(ny)
    X, Y = np.meshgrid(x, y)

    plt.figure(figsize=(7, 6))
    plt.streamplot(X, Y, Ex, Ey,
                   density=1.2, linewidth=1.0, arrowsize=1.2)
    plt.title(title)
    plt.xlabel('x (узел)')
    plt.ylabel('y (узел)')
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.show()


def plot_convergence(history_jacobi, history_gs):
    """График сходимости методов Якоби и Гаусса-Зейделя."""
    plt.figure(figsize=(7, 5))
    if history_jacobi is not None:
        plt.semilogy(history_jacobi, label='Якоби')
    if history_gs is not None:
        plt.semilogy(history_gs, label='Гаусс-Зейдель')
    plt.xlabel('Номер итерации')
    plt.ylabel('max|Δφ|')
    plt.title('Сравнение скорости сходимости')
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


1. Конденсатор без проводящего квадрата.
2. Конденсатор с проводящим квадратом (потенциал квадрата фиксирован).
3. Сравнение числа итераций и темпа убывания невязки
   для методов Якоби и Гаусса–Зейделя.


In [ ]:
nx, ny = 81, 81
V0 = 1.0
tol = 1e-5
max_iters = 20000

phi0, fixed, phi_fix, square_mask = create_parallel_plate_geometry(
    nx=nx, ny=ny, V0=V0, with_square=False
)

phi_gs, it_gs, res_gs, hist_gs = solve_laplace_gauss_seidel(
    phi0, fixed, tol=tol, max_iters=max_iters,
    verbose=True, store_history=True
)

Ex, Ey = compute_field(phi_gs)

plot_potential(phi_gs, title='Потенциал между пластинами без квадрата')
plot_field_quiver(Ex, Ey, title='Поле E без квадрата (quiver)', step=3)
plot_field_stream(Ex, Ey, title='Поле E без квадрата (streamplot)')


phi0_sq, fixed_sq, phi_fix_sq, square_mask_sq = create_parallel_plate_geometry(
    nx=nx, ny=ny, V0=V0,
    with_square=True,
    square_V=0.0,
    square_size=21
)

phi_gs_sq, it_gs_sq, res_gs_sq, hist_gs_sq = solve_laplace_gauss_seidel(
    phi0_sq, fixed_sq, tol=tol, max_iters=max_iters,
    verbose=True, store_history=True
)

Ex_sq, Ey_sq = compute_field(phi_gs_sq)

plot_potential(phi_gs_sq,
               title='Потенциал между пластинами с проводящим квадратом',
               square_mask=square_mask_sq)

plot_field_quiver(Ex_sq, Ey_sq,
                  title='Поле E с проводящим квадратом (quiver)', step=3)

plot_field_stream(Ex_sq, Ey_sq,
                  title='Линии поля E, огибающие проводящий квадрат')


phi0_sq, fixed_sq, phi_fix_sq, square_mask_sq = create_parallel_plate_geometry(
    nx=nx, ny=ny, V0=V0,
    with_square=True,
    square_V=0.0,
    square_size=21
)

# Метод Якоби
phi_jac, it_jac, res_jac, hist_jac = solve_laplace_jacobi(
    phi0_sq, fixed_sq, tol=tol, max_iters=max_iters,
    verbose=True, store_history=True
)

# Метод Гаусса-Зейделя
phi_gs2, it_gs2, res_gs2, hist_gs2 = solve_laplace_gauss_seidel(
    phi0_sq, fixed_sq, tol=tol, max_iters=max_iters,
    verbose=True, store_history=True
)

print('\nИтоги сравнения:')
print(f'  Якоби:         итераций = {it_jac},   итоговая maxΔφ = {res_jac:.3e}')
print(f'  Гаусс-Зейдель: итераций = {it_gs2},   итоговая maxΔφ = {res_gs2:.3e}')

plot_convergence(hist_jac, hist_gs2)
